# Error Analysis: Support Ticket Router

Build a baseline customer-support routing classifier with DSPy, run it on a handful of tickets, then walk through the manual review and dataset construction loop that turns failures into training data.

**Required env vars:** `OPENAI_API_KEY` (loaded from `.env`).

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## A baseline routing program

The signature tells DSPy what the task is. The `Literal` output type constrains the department to one of four allowed labels, and `ChainOfThought` adds a reasoning step before the final classification.

In [ ]:
from typing import Literal

# Your task signature
class SupportRouting(dspy.Signature):
    """Route customer support tickets to the appropriate team."""
    ticket_text: str = dspy.InputField()
    department: Literal["billing", "technical", "sales", "account"] = dspy.OutputField()

# Baseline program (no optimization)
router = dspy.ChainOfThought(SupportRouting)

# Test on a few examples
test_cases = [
    "I can't log into my account after resetting my password",
    "Do you offer enterprise pricing for teams over 100 users?",
    "My credit card was charged twice for the same month",
    "The mobile app crashes when I try to upload images",
]

for ticket in test_cases:
    result = router(ticket_text=ticket)
    print(f"Ticket: {ticket}")
    print(f"Routed to: {result.department}")
    print(f"Reasoning: {result.reasoning}\n")

## A failing edge case

The model treats "can't find the button" as a technical issue, but it is really an account-management question.

In [ ]:
ticket = "I need to cancel my subscription but can't find the button"
result = router(ticket_text=ticket)
print(result.department)  # likely 'technical', should be 'account'

## Annotated examples

Each error gets a short note explaining the gap between the predicted label and the label you actually want. These notes become feedback for an optimizer and reminders for yourself when you revisit the dataset later.

In [ ]:
examples = [
    {
        "ticket": "I need to cancel my subscription but can't find the button",
        "pred_department": "technical",
        "gold_department": "account",
        "notes": "Model sees 'can't find' and assumes technical issue, but this is an account management question",
    },
    {
        "ticket": "Can I downgrade from Pro to Basic plan?",
        "pred_department": "billing",
        "gold_department": "account",
        "notes": "Model associates plan changes with billing, but account team handles subscription changes",
    },
]

for ex in examples:
    print(ex)

## A small evaluation dataset

Group the examples into clear, boundary, and ambiguous cases so each pattern from the error analysis is represented. Then split deterministically into train / val / test.

In [ ]:
dataset = [
    # Clear cases (baseline should get these right)
    dspy.Example(ticket_text="My credit card was declined", department="billing").with_inputs("ticket_text"),
    dspy.Example(ticket_text="The app won't load on iPhone 15", department="technical").with_inputs("ticket_text"),
    dspy.Example(ticket_text="What's your pricing for 500-user teams?", department="sales").with_inputs("ticket_text"),

    # Boundary cases (baseline might fail)
    dspy.Example(ticket_text="I need to cancel my subscription", department="account").with_inputs("ticket_text"),
    dspy.Example(ticket_text="Can I switch from monthly to annual billing?", department="account").with_inputs("ticket_text"),
    dspy.Example(ticket_text="I was charged but my account is still showing as free", department="billing").with_inputs("ticket_text"),

    # Ambiguous cases (hard for any model)
    dspy.Example(ticket_text="I'm locked out", department="account").with_inputs("ticket_text"),
    dspy.Example(ticket_text="How do I upgrade?", department="account").with_inputs("ticket_text"),
    dspy.Example(ticket_text="I can't access the API", department="technical").with_inputs("ticket_text"),
]

print(f"Dataset size: {len(dataset)}")

In [ ]:
import random

# Combine all your examples (already a list of dspy.Example here)
all_examples = list(dataset)

# Shuffle with seed for reproducibility
random.Random(2024).shuffle(all_examples)

# Split into train, val, test
n = len(all_examples)
train_size = int(0.5 * n)
val_size = int(0.25 * n)

train_set = all_examples[:train_size]
val_set = all_examples[train_size:train_size + val_size]
test_set = all_examples[train_size + val_size:]

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

## Documenting genuinely ambiguous tickets

Keep a separate log of the tickets even a human reviewer struggles with. They tell you where the task definition itself is incomplete.

In [ ]:
ambiguous_cases = [
    {
        "ticket": "I'm locked out",
        "needs_clarification": "Could be password reset (technical) or account suspension (account)",
        "default_routing": "account",  # Conservative choice
        "reasoning": "Better to route to account who can triage, rather than technical who can't help with suspensions",
    }
]

for case in ambiguous_cases:
    print(case)